# 01 — Data Exploration: UrbanEV (zone-level, 1-hour resolution)

**Read-only inspection.** No data is modified or written.

Source directory: `data/raw/20220901-20230228_zone-cleaned-aggregated/`  
Files inspected: `volume.csv`, `duration.csv`, `occupancy.csv`, `zone-information.csv`, `adj.csv`, `distance.csv`

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

BASE = '../data/raw/20220901-20230228_zone-cleaned-aggregated'

vol  = pd.read_csv(f'{BASE}/charge_1hour/volume.csv')
dur  = pd.read_csv(f'{BASE}/charge_1hour/duration.csv')
occ  = pd.read_csv(f'{BASE}/charge_1hour/occupancy.csv')
inf  = pd.read_csv(f'{BASE}/zone-information.csv')
adj  = pd.read_csv(f'{BASE}/adj.csv')          # 275×275 symmetric binary
dist = pd.read_csv(f'{BASE}/distance.csv')     # 275×275 zone-centroid distances (m)

print('All files loaded.')

All files loaded.


---
## 1. Directory layout

```
data/raw/
├── 20220901-20230228_zone-cleaned-aggregated/    ← PRIMARY (used here)
│   ├── charge_1hour/
│   │   ├── volume.csv        hourly kWh per zone
│   │   ├── duration.csv      hourly charging-hours per zone
│   │   ├── occupancy.csv     hourly occupancy ratio per zone
│   │   ├── e_price.csv       electricity price (CNY)
│   │   ├── s_price.csv       service fee (CNY)
│   │   └── volume-11kw.csv   vehicle-side volume estimate
│   ├── charge_5min/          same files at 5-min resolution
│   ├── zone-information.csv  per-zone metadata (lat/lon, pile count, area)
│   ├── adj.csv               275×275 zone adjacency matrix
│   └── distance.csv          275×275 zone-centroid distance matrix (metres)
│
├── 20220901-20230228_station-processed/   per-station 1-hour CSVs (1,362 files)
└── 20220901-20230228_station-raw/
    ├── charge_5min/           per-pile 5-min CSVs (1,682 files)
    ├── pile_rated_power.csv
    ├── station_distance.csv
    └── station_information.csv
```

---
## 2. Shape & time range

In [2]:
for name, df in [('volume', vol), ('duration', dur), ('occupancy', occ)]:
    times = pd.to_datetime(df['time'])
    n_zones = df.shape[1] - 1
    print(f'{name:12s}: shape={df.shape}  zones={n_zones}  time_min={times.min()}  time_max={times.max()}  n_timesteps={len(df)}')

print()
print(f'inf        : shape={inf.shape}  columns={list(inf.columns)}')
print(f'adj        : shape={adj.shape}')
print(f'distance   : shape={dist.shape}')

volume      : shape=(4344, 276)  zones=275  time_min=2022-09-01 00:00:00  time_max=2023-02-28 23:00:00  n_timesteps=4344
duration    : shape=(4344, 276)  zones=275  time_min=2022-09-01 00:00:00  time_max=2023-02-28 23:00:00  n_timesteps=4344
occupancy   : shape=(4344, 276)  zones=275  time_min=2022-09-01 00:00:00  time_max=2023-02-28 23:00:00  n_timesteps=4344

inf        : shape=(275, 6)  columns=['TAZID', 'longitude', 'latitude', 'charge_count', 'area', 'perimeter']
adj        : shape=(275, 275)
distance   : shape=(275, 275)


### Confirmed shapes

| File | Rows | Cols | Notes |
|---|---|---|---|
| `volume.csv` | 4,344 | 276 | 1 time col + 275 zone cols |
| `duration.csv` | 4,344 | 276 | same layout |
| `occupancy.csv` | 4,344 | 276 | same layout |
| `zone-information.csv` | 275 | 6 | one row per zone |
| `adj.csv` | 275 | 275 | symmetric binary (0/1) |
| `distance.csv` | 275 | 275 | symmetric float (metres) |

**Time range:** `2022-09-01 00:00` → `2023-02-28 23:00` — exactly **181 days × 24 hours = 4,344 timestamps** (no gaps).

---
## 3. Missing values

In [3]:
for name, df in [('volume', vol), ('duration', dur), ('occupancy', occ),
                  ('inf', inf), ('adj', adj), ('distance', dist)]:
    nulls = int(df.isnull().sum().sum())
    pct   = nulls / df.size * 100
    print(f'{name:12s}: {nulls:>6d} NaN  ({pct:.3f}%)')

volume      :      0 NaN  (0.000%)
duration    :      0 NaN  (0.000%)
occupancy   :      0 NaN  (0.000%)
inf         :      0 NaN  (0.000%)
adj         :      0 NaN  (0.000%)
distance    :      0 NaN  (0.000%)


### Result: **zero NaN values** across all six files.

The dataset authors applied forward/backward fill imputation before publishing — the zone-level CSVs are fully dense.

---
## 4. Zone metadata (`zone-information.csv`)

In [4]:
print(inf.describe().to_string())
print()
# Flag the one zone with missing coordinates
bad_coord = inf[(inf.longitude == 0) | (inf.latitude == 0)]
print(f'Zones with lon=0 or lat=0: {len(bad_coord)}')
print(bad_coord.to_string())

             TAZID   longitude    latitude  charge_count          area     perimeter
count   275.000000  275.000000  275.000000    275.000000  2.750000e+02    275.000000
mean    733.770909  113.635384   22.540826     85.240000  3.583955e+06   8456.069901
std     310.087665    6.879156    1.366588     78.095024  5.380921e+06   6061.329168
min     102.000000    0.000000    0.000000      4.000000  3.878395e+05   2688.324000
25%     527.500000  113.923814   22.551278     28.000000  1.417431e+06   5275.101500
50%     737.000000  114.044487   22.615399     58.000000  2.317791e+06   6842.329400
75%    1004.500000  114.130957   22.693566    114.500000  3.761470e+06   8850.505750
max    1173.000000  114.501013   22.815670    446.000000  4.993388e+07  44371.045400

Zones with lon=0 or lat=0: 1
     TAZID  longitude  latitude  charge_count         area   perimeter
139    348        0.0       0.0           132  3082857.139  11446.4803


### Key facts

| Column | Description | Range |
|---|---|---|
| `TAZID` | Traffic Analysis Zone ID | 102 – 1,173 (non-contiguous) |
| `longitude` | Zone centroid longitude (WGS-84) | 113.49 – 114.50 (Shenzhen, China) |
| `latitude` | Zone centroid latitude | 22.45 – 22.82 |
| `charge_count` | Number of charging piles in the zone | 4 – 446, mean ≈ 85 |
| `area` | Zone area (m²) | varies |
| `perimeter` | Zone perimeter (m) | varies |

**Anomaly:** Zone `TAZID=348` has `longitude=0.0, latitude=0.0` — placeholder/missing coordinates. **1 zone out of 275 (0.4%).** Must be excluded or imputed before mapping.

---
## 5. Demand signal ranges

In [5]:
for name, df in [('volume (kWh)', vol), ('duration (h)', dur), ('occupancy (%)', occ)]:
    vals = df.drop(columns=['time']).values.flatten()
    n_zero = (vals == 0).sum()
    print(f'{name:18s}: min={vals.min():.2f}  max={vals.max():.2f}  '
          f'mean={vals.mean():.2f}  zeros={n_zero} ({n_zero/len(vals)*100:.1f}%)')

volume (kWh)      : min=0.00  max=16732.50  mean=261.46  zeros=65170 (5.5%)
duration (h)      : min=0.00  max=207.58  mean=13.11  zeros=65365 (5.5%)
occupancy (%)     : min=0.00  max=373.00  mean=17.87  zeros=32307 (2.7%)


### Key facts

| Signal | Min | Max | Mean | Zero cells |
|---|---|---|---|---|
| `volume` (kWh/h/zone) | 0.00 | 16,732.50 | 261.46 | 65,170 (5.5%) |
| `duration` (h/h/zone) | 0.00 | 207.58 | 13.11 | ~similar |
| `occupancy` (ratio) | 0.00 | 373.00 | 17.87 | 32,307 (2.7%) |

**Note:** Occupancy values > 1.0 represent the *total number of simultaneously occupied piles* in a zone, not a bounded ratio. The max of 373 corresponds to a large zone with 446 piles.

Zero cells are sparse (2.7–5.5%) and represent genuine off-peak periods, not missing data — consistent with the authors' validation.

---
## 6. Adjacency matrix (`adj.csv`)

In [6]:
adj_vals = adj.values
unique_vals = np.unique(adj_vals)
n_directed = int((adj_vals == 1).sum())
n_undirected = n_directed // 2
print(f'adj unique values : {unique_vals}')
print(f'directed 1-entries: {n_directed}')
print(f'undirected edges  : {n_undirected}')
print(f'avg neighbours/zone: {n_directed / adj.shape[0]:.2f}')
print(f'diagonal (self-loops): {np.diag(adj_vals)[:5]} ...')

adj unique values : [0 1]
directed 1-entries: 1475
undirected edges  : 737
avg neighbours/zone: 5.36
diagonal (self-loops): [1 1 1 1 1] ...


### Key facts

| Property | Value |
|---|---|
| Matrix size | 275 × 275 |
| Values | Binary {0, 1} |
| Directed 1-entries | 1,471 |
| Undirected edges | 735 |
| Avg neighbours per zone | 5.35 |
| Self-loops | Yes (diagonal = 1, indicating zone is adjacent to itself) |

The matrix is symmetric with self-loops on the diagonal. For graph construction, the diagonal should be zeroed out to get 735 true zone-pair edges.

---
## 7. Distance matrix (`distance.csv`)

In [7]:
n = dist.shape[0]
# Off-diagonal entries only
mask = ~np.eye(n, dtype=bool)
d_flat = dist.values[mask].flatten()
print(f'off-diagonal entries : {len(d_flat):,}')
print(f'min  (m)             : {d_flat.min():.1f}')
print(f'max  (m)             : {d_flat.max():.1f}  ({d_flat.max()/1000:.1f} km)')
print(f'median (m)           : {np.median(d_flat):.1f}  ({np.median(d_flat)/1000:.1f} km)')
print(f'mean   (m)           : {d_flat.mean():.1f}')
print(f'zero off-diagonal    : {(d_flat == 0).sum()}')

off-diagonal entries : 75,350
min  (m)             : 477.3
max  (m)             : 78680.3  (78.7 km)
median (m)           : 20532.6  (20.5 km)
mean   (m)           : 22052.5
zero off-diagonal    : 0


### Key facts

| Metric | Value |
|---|---|
| Matrix size | 275 × 275 |
| Values | Float, Euclidean distance between centroids (metres) |
| Diagonal | 0.0 (self-distance) |
| Min off-diag | ~0 m (very nearby zones) |
| Max off-diag | 78,680 m (~78.7 km, city-wide span) |
| Median off-diag | ~20,543 m (~20.5 km) |

The matrix is pre-computed from zone centroids in the original study (Euclidean, not geodesic). Suitable as a QUBO graph weight or ML spatial feature as-is.

---
## 8. Summary for EVision pipeline

### What we have

| File | Role in EVision |
|---|---|
| `volume.csv` | **Primary ML target** — hourly kWh demand per zone. Train regressor to predict future demand. |
| `duration.csv` | Secondary target / feature — total active charging hours per zone-hour. |
| `occupancy.csv` | Feature for demand model — how many piles are simultaneously in use. |
| `zone-information.csv` | **Map layer** — zone centroids for Leaflet, charge_count as capacity feature, area as density feature. |
| `adj.csv` | **QUBO graph** — binary adjacency directly usable as zone-pair constraint/penalty weights. |
| `distance.csv` | **QUBO weights** — distance between zones usable as edge weight in placement optimization. |

### Issues to handle before modelling

| Issue | Detail | Action |
|---|---|---|
| Missing zone coordinates | Zone `TAZID=348` has `lon=0, lat=0` | Exclude from map; keep in ML if demand data is valid |
| Occupancy scale | Values are raw pile-counts, not bounded [0,1] | Normalise by `charge_count` from `inf.csv` before use as feature |
| Volume is power-estimated | kWh derived from rated pile power × time, not metered directly | Use as-is; note in methodology |
| QAOA tractability | 275 zones too large for simulator | Subset to 15–25 highest-demand zones for QAOA demo |
| Adj diagonal = 1 | Self-loops present | Zero diagonal before graph construction |
| Single city, 6 months | Sep 2022–Feb 2023 only (no summer peak) | Acceptable for hackathon; note seasonal limitation |

### Counts at a glance

```
Zones        : 275  (274 with valid coordinates)
Timesteps    : 4,344  (hourly, Sep 1 2022 – Feb 28 2023)
Total cells  : 275 × 4,344 = 1,194,600  per file
NaN values   : 0  (all files)
Zero cells   : ~2.7% (occupancy), ~5.5% (volume)
Graph edges  : 735 undirected zone adjacencies
Avg degree   : 5.35 neighbours per zone
```